# LeaseGuard Colab orchestrator

This notebook is the only place that should download professional benchmarks or run official evaluators. The local laptop keeps code, schemas, and tiny fixtures.

Headline scores come from **LegalBench**, **CUAD**, **ContractNLI**, and **LegalBench-RAG**. The internal lease suite is regression only and cannot support a breakthrough claim.

Set `TASK` to one of: `regression`, `evaluate-legalbench`, `evaluate-cuad`, `evaluate-contractnli`, `evaluate-legalbench-rag`.

In [ ]:
import os
from pathlib import Path

TASK = "regression"
REPO_URL = "https://github.com/YOUR_USER/LeaseGuard.git"
REPO_DIR = Path("/content/LeaseGuard")
CHECKOUT_ROOT = Path("/content/drive/MyDrive/leaseguard/evaluators")
REPORT_DIR = Path("/content/drive/MyDrive/leaseguard/reports")

print({"task": TASK, "repo": str(REPO_DIR)})

In [ ]:
from subprocess import check_call

if not REPO_DIR.exists():
    check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
check_call(["pip", "install", "-e", "."])

In [ ]:
from leaseguard.evaluation.cli import main

REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHECKOUT_ROOT.mkdir(parents=True, exist_ok=True)

if TASK == "regression":
    raise SystemExit(main(["regression", "--output", str(REPORT_DIR / "regression.json")]))

benchmark_id = TASK.removeprefix("evaluate-")
os.environ["LEASEGUARD_ALLOW_BENCHMARK_DOWNLOAD"] = "1"
prepare_status = main(
    [
        "prepare-official",
        "--benchmark",
        benchmark_id,
        "--checkout-root",
        str(CHECKOUT_ROOT),
        "--allow-download",
    ]
)
print("prepare exit", prepare_status)
print("Run the printed git clone and checkout commands, then re-run official-status.")
raise SystemExit(
    main(
        [
            "official-status",
            "--benchmark",
            benchmark_id,
            "--checkout-root",
            str(CHECKOUT_ROOT),
        ]
    )
)